In [4]:
# sample_deep.py
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import time

# 디바이스 확인
print("=== 디바이스 정보 ===")
print("MPS 사용 가능:", torch.backends.mps.is_available())
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print("사용 중인 디바이스:", device)

# 간단한 모델 정의
class SimpleModel(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=256, output_dim=10):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# 더미 데이터 생성
print("\n=== 데이터 생성 ===")
batch_size = 64
input_dim = 100
output_dim = 10
n_samples = 1000

X = torch.randn(n_samples, input_dim)
y = torch.randint(0, output_dim, (n_samples,))
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 모델, 손실 함수, 옵티마이저 설정
print("\n=== 모델 설정 ===")
model = SimpleModel(input_dim=input_dim, output_dim=output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 루프
print("\n=== 학습 시작 ===")
num_epochs = 5
model.train()

for epoch in tqdm(range(num_epochs), desc="Epoch"):
    epoch_loss = 0.0
    epoch_acc = 0.0
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        # 데이터를 디바이스로 이동
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        # 순전파
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # 역전파
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 통계 업데이트
        epoch_loss += loss.item()
        epoch_acc += (outputs.argmax(dim=1) == targets).float().mean().item()
        
        # 배치마다 진행상황 출력
        if batch_idx % 10 == 0:
            print(f"\nBatch {batch_idx}:")
            print(f"Loss: {loss.item():.4f}")
            print(f"Accuracy: {(outputs.argmax(dim=1) == targets).float().mean().item():.4f}")
    
    # 에폭마다 평균 통계 출력
    print(f"\nEpoch {epoch+1} 완료:")
    print(f"평균 Loss: {epoch_loss/len(dataloader):.4f}")
    print(f"평균 Accuracy: {epoch_acc/len(dataloader):.4f}")

print("\n=== 학습 완료 ===")

=== 디바이스 정보 ===
MPS 사용 가능: True
사용 중인 디바이스: mps

=== 데이터 생성 ===

=== 모델 설정 ===

=== 학습 시작 ===


Epoch:  40%|████      | 2/5 [00:00<00:00, 13.38it/s]


Batch 0:
Loss: 2.3163
Accuracy: 0.1094

Batch 10:
Loss: 2.3071
Accuracy: 0.0781

Epoch 1 완료:
평균 Loss: 2.3081
평균 Accuracy: 0.0990

Batch 0:
Loss: 2.1911
Accuracy: 0.3281

Batch 10:
Loss: 2.2085
Accuracy: 0.1250

Epoch 2 완료:
평균 Loss: 2.2024
평균 Accuracy: 0.1744

Batch 0:
Loss: 2.1597
Accuracy: 0.3594

Batch 10:
Loss: 2.1239
Accuracy: 0.2500


Epoch: 100%|██████████| 5/5 [00:00<00:00, 12.73it/s]


Epoch 3 완료:
평균 Loss: 2.0921
평균 Accuracy: 0.3893

Batch 0:
Loss: 2.0198
Accuracy: 0.3906

Batch 10:
Loss: 1.8675
Accuracy: 0.5469

Epoch 4 완료:
평균 Loss: 1.9377
평균 Accuracy: 0.4236

Batch 0:
Loss: 1.8393
Accuracy: 0.4688

Batch 10:
Loss: 1.7239
Accuracy: 0.4531

Epoch 5 완료:
평균 Loss: 1.7398
평균 Accuracy: 0.4807

=== 학습 완료 ===


In [1]:
import pandas as pd

In [3]:
test = pd.read_csv('deep-submission.csv')
test['Segment'] = test['target']
test.drop(columns=['target'], inplace=True)
test.to_csv('deep-submission.csv', index=False)